# 📘 แบบฝึกหัดสร้างฐานข้อมูล MySQL และนำเข้าข้อมูลแบบเข้าใจง่าย (Beginner Guide)
### โครงงาน: World Bank Commodity Price (แบบฉบับนักศึกษาเริ่มต้น)
สมุดงานนี้จำลองการทำงานด้วยโค้ด Python แบบเรียบง่าย โดยใช้ไลบรารีมาตรฐาน **`mysql.connector`** ตามแนวทางจากเอกสารทางการของ MySQL


## 1. นำเข้าไลบรารีและตั้งค่าการเชื่อมต่อ (Setup & Connection)


In [ ]:
import os
import sys
import pandas as pd
import mysql.connector
from mysql.connector import errorcode

# กำหนดค่าการเชื่อมต่อ MySQL (XAMPP Default)
CONFIG = {
    'user': 'root',
    'password': '',
    'host': '127.0.0.1',
    'port': 3306,
    'raise_on_warnings': False
}

DB_NAME = 'world_bank_simple'

print("ตั้งค่าพร้อมเชื่อมต่อฐานข้อมูล:", DB_NAME)


## 2. กำหนดโครงสร้างตารางด้วย Dictionary (อิงตามตัวอย่าง mysql.py)
เราจะสร้าง 2 ตารางเชื่อมโยงกันด้วย Foreign Key (1-to-Many Relationship):
1. **`dim_commodity`**: ตารางมิติสินค้า เก็บชื่อสินค้า 71 รายการ ไม่ให้มีข้อมูลซ้ำ
2. **`fact_monthly_prices`**: ตารางบันทึกราคารายเดือน เชื่อมโยงผ่าน `commodity_id`


In [ ]:
TABLES = {}

# 1) ตาราง Dimension: ข้อมูลสินค้า
TABLES['dim_commodity'] = (
    "CREATE TABLE `dim_commodity` ("
    "  `commodity_id` int(11) NOT NULL AUTO_INCREMENT,"
    "  `commodity_name` varchar(100) NOT NULL,"
    "  `group_product` varchar(100) NOT NULL,"
    "  `unit` varchar(50) DEFAULT NULL,"
    "  `source` text DEFAULT NULL,"
    "  `description` text DEFAULT NULL,"
    "  `created_at` timestamp DEFAULT CURRENT_TIMESTAMP,"
    "  PRIMARY KEY (`commodity_id`),"
    "  UNIQUE KEY `commodity_name` (`commodity_name`)"
    ") ENGINE=InnoDB DEFAULT CHARSET=utf8mb4")

# 2) ตาราง Fact: ข้อมูลราคารายเดือน
TABLES['fact_monthly_prices'] = (
    "CREATE TABLE `fact_monthly_prices` ("
    "  `price_id` int(11) NOT NULL AUTO_INCREMENT,"
    "  `date` date NOT NULL,"
    "  `commodity_id` int(11) NOT NULL,"
    "  `price` decimal(12,4) DEFAULT NULL,"
    "  `created_at` timestamp DEFAULT CURRENT_TIMESTAMP,"
    "  PRIMARY KEY (`price_id`),"
    "  KEY `commodity_id` (`commodity_id`),"
    "  CONSTRAINT `fact_monthly_prices_fk_1` FOREIGN KEY (`commodity_id`) "
    "     REFERENCES `dim_commodity` (`commodity_id`) ON DELETE CASCADE"
    ") ENGINE=InnoDB DEFAULT CHARSET=utf8mb4")

print("นิยามโครงสร้างตาราง TABLES เรียบร้อยแล้ว!")


## 3. สร้างฐานข้อมูลและตารางบน MySQL


In [ ]:
# เชื่อมต่อ MySQL Server
cnx = mysql.connector.connect(**CONFIG)
cursor = cnx.cursor()

# 1. ตรวจสอบและสร้างฐานข้อมูล
try:
    cursor.execute(f"USE {DB_NAME}")
    print(f"เชื่อมต่อฐานข้อมูล {DB_NAME} สำเร็จ")
except mysql.connector.Error as err:
    if err.errno == errorcode.ER_BAD_DB_ERROR:
        cursor.execute(f"CREATE DATABASE {DB_NAME} DEFAULT CHARACTER SET 'utf8mb4'")
        print(f"สร้างฐานข้อมูล {DB_NAME} สำเร็จ!")
        cnx.database = DB_NAME
    else:
        print(err)

# 2. วนลูปสร้างตารางจาก Dictionary TABLES
for table_name in TABLES:
    table_sql = TABLES[table_name]
    try:
        print(f"กำลังสร้างตาราง {table_name}... ", end="")
        cursor.execute(table_sql)
        print("OK")
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_TABLE_EXISTS_ERROR:
            print("ตารางนี้มีอยู่แล้ว (Already exists)")
        else:
            print(err.msg)


## 4. นำเข้าข้อมูล Dimension (`dim_commodity`) จากไฟล์ CSV


In [ ]:
# อ่านไฟล์ dim_commodity.csv
df_dim = pd.read_csv("dim_commodity.csv")
print("จำนวนสินค้าในไฟล์:", len(df_dim))

insert_dim_sql = (
    "INSERT IGNORE INTO `dim_commodity` "
    "(`commodity_name`, `group_product`, `unit`, `source`, `description`) "
    "VALUES (%s, %s, %s, %s, %s)"
)

dim_data = []
for _, row in df_dim.iterrows():
    unit_val = None if pd.isna(row['unit']) else str(row['unit'])
    source_val = None if pd.isna(row['source']) else str(row['source'])
    desc_val = None if pd.isna(row['description']) else str(row['description'])
    dim_data.append((
        row['commodity_name'],
        row['group_product'],
        unit_val,
        source_val,
        desc_val
    ))

cursor.executemany(insert_dim_sql, dim_data)
cnx.commit()
print(f"โหลดข้อมูลสินค้าเข้าตาราง dim_commodity สำเร็จ! (มีข้อมูลแล้ว {len(dim_data)} รายการ)")


## 5. นำเข้าข้อมูล Fact (`fact_monthly_prices`) จาก clean_df


In [ ]:
# 1. ดึง commodity_id มาทำเป็นพจนานุกรมเพื่อใช้จับคู่
cursor.execute("SELECT commodity_name, commodity_id FROM dim_commodity")
comm_dict = dict(cursor.fetchall())
print(f"ดึงรหัสสินค้ามาได้ {len(comm_dict)} รายการ เช่น 'Aluminum' -> ID {comm_dict.get('Aluminum')}")

# 2. อ่านไฟล์ clean_df (monthly_prices_cleaned.csv)
df_clean = pd.read_csv("../data/processed/monthly_prices_cleaned.csv")
print(f"จำนวนแถวข้อมูลราคาที่จะนำเข้า: {len(df_clean):,} แถว")

# เคลียร์ข้อมูลเดิมก่อนโหลดชุดใหม่
cursor.execute("DELETE FROM fact_monthly_prices")
cnx.commit()

# 3. เตรียมข้อมูล
insert_fact_sql = (
    "INSERT INTO `fact_monthly_prices` (`date`, `commodity_id`, `price`) "
    "VALUES (%s, %s, %s)"
)

records = []
for _, row in df_clean.iterrows():
    cid = comm_dict.get(row['Commodity'])
    d_str = str(row['Date'])[:10]
    p_val = None if pd.isna(row['Price']) else float(row['Price'])
    records.append((d_str, cid, p_val))

# 4. Insert ข้อมูลแบบทีละ Batch (Batch ละ 5,000 แถว)
batch_size = 5000
for i in range(0, len(records), batch_size):
    batch = records[i:i + batch_size]
    cursor.executemany(insert_fact_sql, batch)
    cnx.commit()

print("โหลดข้อมูล Fact สำเร็จเรียบร้อยครบทุกแถว!")


## 6. ตรวจสอบข้อมูลด้วยคำสั่ง SQL JOIN พื้นฐาน


In [ ]:
query = """
SELECT 
    f.date,
    d.commodity_name,
    d.group_product,
    d.unit,
    f.price
FROM fact_monthly_prices f
JOIN dim_commodity d ON f.commodity_id = d.commodity_id
WHERE f.price IS NOT NULL
ORDER BY f.date DESC, f.price DESC
LIMIT 10;
"""

cursor.execute(query)
results = cursor.fetchall()

# แสดงผลด้วยตาราง Pandas อย่างสวยงาม
df_result = pd.DataFrame(results, columns=['Date', 'Commodity', 'Group', 'Unit', 'Price'])
display(df_result)

# ปิดการเชื่อมต่อ
cursor.close()
cnx.close()
print("ปิดการเชื่อมต่อเรียบร้อย!")
